
# 🧪 Mini Lab: Word2Vec (Skip‑Gram with Negative Sampling)

**Goal:** Understand the *idea* behind Word2Vec by **only** implementing three small pieces of logic using native Python data structures (lists & dicts):

1. Initialize two sets of word embeddings (target and context) as Python lists.
2. Sample **positive** context words around a target word (window-based).
3. Sample **negative** words (simple random words not in the positive context).

Then you'll use a **pre-written PyTorch model** (a "black box") that takes your preprocessed data to train Word2Vec.  
Finally, you'll **experiment with hyperparameters** and write two observations.

---

## Learning Objectives

By the end of this mini lab, you should be able to:

- Explain the **skip‑gram** idea (predicting context from a target).
- Implement simple **window-based context sampling**.
- Implement simple **negative sampling** with pure Python.
- Train a **Word2Vec** model and explore **cosine similarity** between words.
- Observe how **hyperparameters** (embedding size, negatives per positive, epochs) affect results.



## 0. Setup

We’ll keep this mini-lab lightweight and **CPU-runnable**. The next cell will install the two packages we need:

- **PyTorch** — for the Word2Vec model and training loop  
- **NLTK** — for basic tokenization

We will use a very small Shakespeare sample called `tiny_shakespeare_sample.txt`. If the provided file is saved elsewhere, update `DATA_PATH`.

> ✅ **Note:** We'll attempt to use NLTK's `word_tokenize`. If the tokenizer is not installed, the code will try to download it. If that also fails (e.g., you're offline), a simple regex fallback tokenizer will be used.


In [2]:
# If your environment blocks %pip, run these in a terminal instead.
%pip -q install torch nltk

^C
Note: you may need to restart the kernel to use updated packages.


In [1]:

# Standard library
import os, re, math, random, collections
from collections import Counter, defaultdict
from typing import List, Dict, Tuple

# Third-party libraries
try:
    import nltk
    from nltk.tokenize import word_tokenize
except Exception as e:
    nltk = None
    word_tokenize = None

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Reproducibility
random.seed(42)

DATA_PATH = "tiny_shakespeare_sample.txt"

# Make sure the input corpus exists
assert os.path.exists(DATA_PATH)

/usr/local/python/3.12.1/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))



## 1. Load & tokenize the text

In this step we turn the raw file at `DATA_PATH` into a list of tokens called `tokens`. We do the following things:

- Lowercase everything.
- Tokenize words using **NLTK** if available; otherwise, use a simple regex fallback.


In [2]:

def ensure_nltk_tokenizer():
    """Ensure NLTK tokenizer is available; download if needed.
    Returns word_tokenize function or None if failed.
    """
    global nltk, word_tokenize
    if nltk is None:
        try:
            import nltk
            from nltk.tokenize import word_tokenize
        except Exception:
            return None
    try:
        import nltk
        # NLTK 3.8+ sometimes needs both 'punkt' and 'punkt_tab'
        try:
            nltk.data.find('tokenizers/punkt')
        except LookupError:
            nltk.download('punkt', quiet=True)
        try:
            nltk.data.find('tokenizers/punkt_tab')
        except LookupError:
            try:
                nltk.download('punkt_tab', quiet=True)
            except Exception:
                pass
        from nltk.tokenize import word_tokenize
        return word_tokenize
    except Exception:
        return None


def simple_regex_tokenize(text: str) -> List[str]:
    # letters, digits, and apostrophes; split on everything else
    tokens = re.findall(r"[a-zA-Z0-9']+", text.lower())
    return tokens


def load_and_tokenize(path: str) -> List[str]:
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read().lower()
    wt = ensure_nltk_tokenizer()
    if wt is not None:
        try:
            return wt(text)
        except Exception:
            pass
    # Fallback
    return simple_regex_tokenize(text)


tokens = load_and_tokenize(DATA_PATH)
print(f"Total tokens: {len(tokens)}\nSample: {tokens[:30]}" )


Total tokens: 204089
Sample: ['first', 'citizen', 'before', 'we', 'proceed', 'any', 'further', 'hear', 'me', 'speak', 'all', 'speak', 'speak', 'first', 'citizen', 'you', 'are', 'all', 'resolved', 'rather', 'to', 'die', 'than', 'to', 'famish', 'all', 'resolved', 'resolved', 'first', 'citizen']



## 2. Build the vocabulary

We'll map each unique word to an integer ID. For speed in class, you may cap tokens or use a minimum frequency.

## 2. Build the vocabulary

In this step we turn the token list from §1 into a compact vocabulary and two lookup tables:
- `word_to_idx: dict[str, int]` — maps each kept token to a unique integer ID.
- `idx_to_word: list[str]` — reverse lookup by index (length = vocabulary size).

**Why we cap/threshold the vocab**
To keep training **fast on CPU**, we (a) drop very rare tokens and (b) cap the total number of tokens kept:
- `MIN_FREQ = 3` (default): only keep words that appear **at least 3 times**.
- `MAX_TOKENS = 20_000` (default): keep first **20k** tokens.

> If you’re on a **slow machine**, consider **increasing** `MIN_FREQ` (e.g., 5 or 10) and **decreasing** `MAX_TOKENS` (e.g., 10_000).  
> This reduces the embedding size and speeds up training at the cost of vocabulary coverage.

**Effects on speed/quality**
- Smaller vocab → **faster** training and less memory.
- But overly aggressive filtering may **remove informative words** and reduce embedding quality.
- If a query word isn’t kept, later lookups will report “not in vocabulary.”

In [3]:

def build_vocab(tokens: List[str], min_freq: int = 1, max_tokens: int = None):
    if max_tokens is not None:
        tokens_use = tokens[:max_tokens]
    else:
        tokens_use = tokens
    freqs = Counter(tokens_use)
    vocab = [w for w, c in freqs.items() if c >= min_freq]
    vocab.sort()
    word_to_idx = {w: i for i, w in enumerate(vocab)}
    idx_to_word = {i: w for w, i in word_to_idx.items()}
    return vocab, word_to_idx, idx_to_word, freqs

# You can tweak these for speed on slower machines.
MIN_FREQ = 3
MAX_TOKENS = 20_000

vocab, word_to_idx, idx_to_word, freqs = build_vocab(tokens, MIN_FREQ, MAX_TOKENS)
print(f"Vocab size: {len(vocab)}\nSample: {vocab[:20]}" )


Vocab size: 870
Sample: ["'", "'em", "'gainst", "'i", "'s", "'shall'", "'t", "'tis", "'twas", "'twere", "'twixt", 'a', 'about', 'absence', 'absolute', 'account', 'act', 'action', 'aedile', 'aediles']


## 3. 🧩 Your small implementations (pure Python)

You will implement **three** helper functions using **native Python** (lists & dicts; no NumPy/torch).  
These live in the **Section 3** TODO cells, and they are used immediately in §4 (dataset building) and §6 (training).

**What you’ll produce**
- `init_embeddings(...)` → two Python dicts that act like tiny embedding tables.
- `get_positive_context(...)` → windowed context words around a center position.
- `sample_negative_words(...)` → a few “fake” context words (negatives) that are *not* in the true window.

### 3.1 Initialize embeddings

Create **two** embedding tables as Python dictionaries:  
- `target_embeddings`: maps each word → a list of floats (length = `embed_dim`)  
- `context_embeddings`: same as above (independent parameters)

Use small random values (e.g., uniform in \([-0.5/\text{embed\_dim},\;0.5/\text{embed\_dim}]\)).  
Please use a provided `seed` for reproducibility.

**Inputs**
- `vocab`: list of unique tokens (from §2).
- `embed_dim`: length of each embedding vector (e.g., 50).
- `seed`: random seed for reproducibility.

**Output**
- A pair `(target_embeddings, context_embeddings)`, each a `dict[str, list[float]]`.

In [ ]:

def init_embeddings(vocab: List[str], embed_dim: int = 50, seed: int = 42):
    """Return (target_embeddings, context_embeddings) as dicts:
    { word: [float, float, ... length embed_dim ...] }
    """
    random.seed(seed)

    target_embeddings = {}
    low, high = -0.5/embed_dim, 0.5/embed_dim

    for word in vocab:
        tgt = []
        for i in range(embed_dim):
            tgt.append(random.uniform(low, high))
        target_embeddings[word] = tgt
    
    context_embeddings = {}
    for word in vocab:
        con = []
        for i in range(embed_dim):
            con.append(random.uniform(low, high))
        context_embeddings[word] = con

    return (target_embeddings, context_embeddings)    

In [5]:
# Inline tests for init_embeddings
t_emb, c_emb = init_embeddings(['romeo','juliet'], embed_dim=4, seed=7)
assert set(t_emb.keys()) == {'romeo','juliet'} and set(c_emb.keys()) == {'romeo','juliet'}
assert len(t_emb['romeo']) == 4 and len(c_emb['juliet']) == 4
# Ranges should be small
assert all(-0.2 <= x <= 0.2 for x in t_emb['romeo'])
# Different dicts (not same object / values)
assert t_emb['romeo'] != c_emb['romeo']
print("✅ init_embeddings tests passed")

✅ init_embeddings tests passed



### 3.2 Sample **positive** context words (window-based)

**Goal**  
Return the list of **context words** around `tokens[i]` within a symmetric window, **excluding** the center word itself. Order should be left‑to‑right as they appear in the text.

**Inputs**
- `tokens`: the full tokenized corpus from §1.
- `i`: the index of the **center** token in `tokens`.
- `window_size`: how many tokens to take on each side (left and right).

**Output**
- A `List[str]` of context tokens within the window around `i`, **excluding** `tokens[i]`.

**Rules / Hints**
- Preserve **order** (left to right).
- If `i` is near the beginning or end, the window will be truncated (that’s fine).

**Sanity checks**
- `get_positive_context(['a','b','c','d','e'], i=2, window_size=1)  → ['b','d']`
- `get_positive_context(['a','b','c','d','e'], i=0, window_size=2)  → ['b','c']`
- `get_positive_context(['a','b','c','d','e'], i=4, window_size=2)  → ['c','d']`


In [ ]:

def get_positive_context(tokens: List[str], i: int, window_size: int) -> List[str]:
    """Return a list of context words within the window around index i (exclude tokens[i])."""
    context = []
    if i == 0:
        endLoop = window_size + 1 if window_size + 1 < len(tokens) else len(tokens)
        for j in range(1, endLoop):
            context.append(tokens[j])
    elif i == len(tokens) - 1:
        for k in range(i - window_size, i):
            context.append(tokens[k])
    else:
        low, high = i, i
        while window_size > 0:
            low -= 1
            high += 1
            if low >= 0: context.append(tokens[low])
            if high < len(tokens): context.append(tokens[high])
            window_size -= 1

    return context    

In [7]:
# Inline tests for get_positive_context
toks = ['a','b','c','d','e']
assert get_positive_context(toks, 2, 1) == ['b','d']
assert get_positive_context(toks, 0, 2) == ['b','c']
assert get_positive_context(toks, 4, 2) == ['c','d']
print("✅ get_positive_context tests passed")

✅ get_positive_context tests passed



### 3.3 Sample **negative** words (simple random)

Return `num_negatives` words **not** in the positive context and **not** the target itself.  
Do **not** overthink: just sample uniformly at random from the remaining vocabulary.

> Implementation tip: Use `random.sample` for **unique** negatives. If there are not enough candidates (very small toy case), sample as many as you can.

**Inputs**
- `vocab`: list of unique tokens from §2.
- `positives_set`: a `set[str]` of true context words for the current center (from §3.2).
- `target_word`: the current center token.
- `num_negatives`: how many negatives to draw.
- `seed`: random seed for reproducibility.

**Output**
- A `List[str]` of **unique** negatives, length up to `num_negatives` (or fewer if not enough candidates).


In [ ]:

def sample_negative_words(
    vocab: List[str],
    positives_set: set,
    target_word: str,
    num_negatives: int,
    seed: int = 42
) -> List[str]:
    """Return a list of *unique* negative words sampled from vocab,
    excluding positives and the target itself.
    """
    
    candidates = []
    for word in vocab:
        if word not in positives_set and word != target_word:
            candidates.append(word)
    
    k = num_negatives if len(candidates) >= num_negatives else len(candidates)
    return random.Random(seed).sample(candidates, k) 

In [9]:
# Inline tests for sample_negative_words
vocab_small = ['a','b','c','d','e','x','y']
pos_set = {'b','d'}
negs = sample_negative_words(vocab_small, pos_set, 'c', 3, seed=123)
assert all(w not in pos_set and w != 'c' for w in negs)
assert len(set(negs)) == len(negs)  # unique
negs2 = sample_negative_words(vocab_small, pos_set, 'c', 3, seed=123)
assert negs == negs2  # deterministic by seed
print("✅ sample_negative_words tests passed")

✅ sample_negative_words tests passed



## 4. Build training instances from your functions

Using your three functions above, create all **(target, positive, negatives)** instances for the corpus.

- For each token position `i` in `tokens`:
  - Find its positive context words with `get_positive_context`.
  - For each positive word, draw `num_negatives` negatives with `sample_negative_words`.
- Convert words to integer IDs using `word_to_idx`.

We'll return three aligned lists of integer IDs: `centers`, `positives`, and `negatives` (2D list where each row has `num_negatives` IDs).


In [10]:

def build_training_data(
    tokens: List[str],
    word_to_idx: Dict[str, int],
    vocab: List[str],
    window_size: int = 2,
    num_negatives: int = 5,
    seed: int = 123
) -> Tuple[List[int], List[int], List[List[int]]]:
    centers: List[int] = []
    positives: List[int] = []
    negatives: List[List[int]] = []
    rng = random.Random(seed)

    for i, target_word in enumerate(tokens):
        pos_words = get_positive_context(tokens, i, window_size)
        if not pos_words:
            continue
        pos_set = set(pos_words)
        for pw in pos_words:
            # Sample negatives freshly for each (target, positive) pair
            # Use a derived seed so the run is reproducible but varies by pair
            negs = sample_negative_words(
                vocab=vocab,
                positives_set=pos_set,
                target_word=target_word,
                num_negatives=num_negatives,
                seed=rng.randint(0, 10**9)
            )
            # Filter pairs that include OOV (shouldn't happen if vocab from tokens)
            if (target_word in word_to_idx) and (pw in word_to_idx):
                centers.append(word_to_idx[target_word])
                positives.append(word_to_idx[pw])
                negatives.append([word_to_idx[nw] for nw in negs])
    return centers, positives, negatives


# Quick dry run
centers_demo, positives_demo, negatives_demo = build_training_data(
    tokens, word_to_idx, vocab, window_size=2, num_negatives=3, seed=2025
)
print(f"Built {len(centers_demo)} training pairs (each with {len(negatives_demo[0]) if negatives_demo else 0} negatives)." )


Built 432260 training pairs (each with 3 negatives).


## Worked example: one step by hand (read this before running the cell)

This tiny example shows, **on a toy sequence**, how your functions from §3 behave together:
- We pick a **center index** `i = 2` (the second `"romeo"` in the list).
- We set a **window size** of `2`, which collects up to 2 tokens on the **left** and **right** of the center.
- We call `get_positive_context(...)` to get the true **positive context** words around the center (excluding the center itself).
- We call `sample_negative_words(...)` to draw a few **negative** words from the vocabulary **that are not in the positive set and are not the target**.

**What the code prints**
- `Center:` the center token at position `i`  
- `Positives:` the list returned by `get_positive_context` (left→right order).  
  On this toy sequence, because `"romeo"` occurs twice, it’s normal to see `"romeo"` appear in the positives if it’s within the window.  
- `Negatives:` the list from `sample_negative_words`. With a **small vocabulary**, you may get **fewer negatives** than requested (e.g., only 1) — that’s expected, because we exclude the center and all positives from the candidate pool.

In [11]:
# Worked example: one step by hand
example_tokens = ["o", "romeo", "romeo", "wherefore", "art", "thou"]
i = 2  # center = second "romeo"
window = 2
pos = get_positive_context(example_tokens, i, window)
neg = sample_negative_words(sorted(set(example_tokens)), set(pos), example_tokens[i], num_negatives=3, seed=0)
print("Center:", example_tokens[i])
print("Positives:", pos)
print("Negatives:", neg)

Center: romeo
Positives: ['romeo', 'wherefore', 'o', 'art']
Negatives: ['thou']



## 5. ✅ Sanity tests (so you know you're on track)

Run the tests below. If everything is correct, you'll see **All tests passed!**  
If a test fails, read the error message and check your TODO implementations above.


In [12]:

def run_sanity_tests():
    # 1) Test positive sampling on a tiny sequence
    toks = ['a','b','c','d','e']
    assert get_positive_context(toks, 2, 1) == ['b','d']
    assert get_positive_context(toks, 0, 2) == ['b','c']
    assert get_positive_context(toks, 4, 2) == ['c','d']

    # 2) Test negative sampling excludes positives & target
    vocab_small = ['a','b','c','d','e','x','y']
    pos_set = set(['b','d'])
    negs = sample_negative_words(vocab_small, pos_set, 'c', 3, seed=123)
    for w in negs:
        assert w not in pos_set and w != 'c'
    assert len(set(negs)) == len(negs)  # unique

    # 3) Test embedding init shape/values
    v = ['cat','dog']
    t_emb, c_emb = init_embeddings(v, embed_dim=4, seed=7)
    assert set(t_emb.keys()) == set(v) and set(c_emb.keys()) == set(v)
    assert all(len(vec) == 4 for vec in t_emb.values())
    assert all(len(vec) == 4 for vec in c_emb.values())
    # Values should be small
    assert all(all(abs(x) < 0.5 for x in vec) for vec in t_emb.values())

    # 4) Test build_training_data aligns outputs
    vocab_tiny = ['a','b','c','d','e']
    w2i = {w:i for i,w in enumerate(vocab_tiny)}
    centers, positives, negatives = build_training_data(
        toks, w2i, vocab_tiny, window_size=1, num_negatives=2, seed=42
    )
    assert len(centers) == len(positives) == len(negatives)
    assert all(len(row) <= 2 for row in negatives)  # Might be fewer if small candidate set

    print("All tests passed! ✅")


run_sanity_tests()


All tests passed! ✅



## 6. Black‑box: PyTorch Word2Vec with Negative Sampling

You will **not** need to modify this code. It takes the training triples you created and trains a small skip‑gram Word2Vec model with negative sampling.

### What it does (high level)
For each pair `(center, positive)` and `K` negatives:
- Maximizes \(\log \sigma(\mathbf{v}_c^\top \mathbf{v}'_p)\)
- Maximizes \(\sum_{k=1}^K \log \sigma(-\mathbf{v}_c^\top \mathbf{v}'_{n_k})\)

Where \(\mathbf{v}\) are **target (input)** embeddings and \(\mathbf{v}'\) are **context (output)** embeddings.


In [13]:

# If PyTorch isn't available, give a friendly message.
if torch is None:
    raise ImportError(
        "PyTorch is required for the training section. Install with: `pip install torch`"
    )

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class SkipGramNegDataset(Dataset):
    def __init__(self, centers: List[int], positives: List[int], negatives: List[List[int]]):
        assert len(centers) == len(positives) == len(negatives)
        self.centers = centers
        self.positives = positives
        self.negatives = negatives

    def __len__(self):
        return len(self.centers)

    def __getitem__(self, idx):
        c = self.centers[idx]
        p = self.positives[idx]
        neg = self.negatives[idx]
        return (
            torch.tensor(c, dtype=torch.long),
            torch.tensor(p, dtype=torch.long),
            torch.tensor(neg, dtype=torch.long)
        )

class Word2VecNegSampling(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 50):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.input_emb = nn.Embedding(vocab_size, embed_dim)   # target
        self.output_emb = nn.Embedding(vocab_size, embed_dim)  # context
        # Initialize small random values like the classic Word2Vec setup
        init_range = 0.5 / embed_dim
        nn.init.uniform_(self.input_emb.weight, -init_range, init_range)
        nn.init.uniform_(self.output_emb.weight, -init_range, init_range)

    def forward(self, center_ids: torch.Tensor, pos_ids: torch.Tensor, neg_ids: torch.Tensor):
        # center_ids: (B,)
        # pos_ids:    (B,)
        # neg_ids:    (B, K)
        v_c = self.input_emb(center_ids)            # (B, D)
        v_p = self.output_emb(pos_ids)              # (B, D)
        v_n = self.output_emb(neg_ids)              # (B, K, D)

        # Positive score: dot(center, pos)
        pos_score = torch.sum(v_c * v_p, dim=1)     # (B,)
        pos_loss = F.logsigmoid(pos_score)          # (B,)

        # Negative score: dot(center, neg) for each negative
        # (B, K, D) · (B, D, 1) -> (B, K, 1) -> (B, K)
        neg_score = torch.bmm(v_n, v_c.unsqueeze(2)).squeeze(2)
        neg_loss = F.logsigmoid(-neg_score).sum(dim=1)  # (B,)

        loss = -(pos_loss + neg_loss).mean()
        return loss

    def get_input_vector(self, idx: int) -> torch.Tensor:
        return self.input_emb.weight[idx].detach()

    def cosine_similarity(self, idx1: int, idx2: int) -> float:
        v1 = self.get_input_vector(idx1)
        v2 = self.get_input_vector(idx2)
        num = torch.dot(v1, v2).item()
        den = (v1.norm().item() * v2.norm().item()) or 1e-8
        return float(num / den)


class Word2VecTrainer:
    def __init__(self, vocab: List[str], word_to_idx: Dict[str, int], embed_dim: int = 50):
        self.vocab = vocab
        self.word_to_idx = word_to_idx
        self.idx_to_word = {i: w for w, i in word_to_idx.items()}
        self.model = Word2VecNegSampling(vocab_size=len(vocab), embed_dim=embed_dim).to(device)

    def train(self, centers, positives, negatives, epochs=2, batch_size=256, lr=0.025, verbose=True):
        dataset = SkipGramNegDataset(centers, positives, negatives)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=False)
        opt = torch.optim.Adam(self.model.parameters(), lr=lr)

        for ep in range(1, epochs+1):
            total_loss, n = 0.0, 0
            for c, p, nneg in loader:
                c, p, nneg = c.to(device), p.to(device), nneg.to(device)
                opt.zero_grad()
                loss = self.model(c, p, nneg)
                loss.backward()
                opt.step()
                total_loss += loss.item() * c.size(0)
                n += c.size(0)
            if verbose:
                print(f"Epoch {ep}/{epochs} — avg loss: {total_loss / max(n,1):.4f}")

    def similarity(self, w1: str, w2: str) -> float:
        """Cosine similarity between two *words* (strings)."""
        if (w1 not in self.word_to_idx) or (w2 not in self.word_to_idx):
            raise ValueError("Both words must be in the vocabulary.")
        i1, i2 = self.word_to_idx[w1], self.word_to_idx[w2]
        return self.model.cosine_similarity(i1, i2)

    def most_similar(self, word: str, top_k: int = 10) -> List[Tuple[str, float]]:
        if word not in self.word_to_idx:
            raise ValueError(f"'{word}' not in vocabulary.")
        target_idx = self.word_to_idx[word]
        sims = []
        for i in range(len(self.vocab)):
            if i == target_idx:
                continue
            s = self.model.cosine_similarity(target_idx, i)
            sims.append((self.idx_to_word[i], s))
        sims.sort(key=lambda x: x[1], reverse=True)
        return sims[:top_k]



## 7. Train the model (quick demo)

This is a **small** training run. Feel free to increase epochs, embedding size, or negatives later.

> Notes:
> - If training is slow, reduce `MAX_TOKENS` earlier or lower `epochs`.
> - Cosine similarity values are **not** guaranteed to match any specific threshold—just use them relatively to compare words.


In [16]:

# Build training data (you can tweak these)
WINDOW_SIZE = 2
NEGATIVES_PER_POS = 5
EMBED_DIM = 50
EPOCHS = 2
BATCH_SIZE = 256
LR = 0.025

centers, positives, negatives = build_training_data(
    tokens, word_to_idx, vocab,
    window_size=WINDOW_SIZE,
    num_negatives=NEGATIVES_PER_POS,
    seed=2025
)
print(f"Training pairs: {len(centers)}; each with {NEGATIVES_PER_POS} negatives (or fewer in tiny vocab cases)." )

trainer = Word2VecTrainer(vocab, word_to_idx, embed_dim=EMBED_DIM)
trainer.train(centers, positives, negatives, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, verbose=True)

# Try a couple of cosine similarities if words exist
test_pairs = [
    ('love','hate'), ('good','bad'), ('poor','rich'), ('all', 'none'),
    ('nay', 'no'), ('valiant', 'brave'), ('say', 'tell'), ('small', 'little')
]
for w1, w2 in test_pairs:
    if w1 in word_to_idx and w2 in word_to_idx:
        sim = trainer.similarity(w1, w2)
        print(f"cos('{w1}', '{w2}') = {sim:.3f}")
    else:
        print(f"(Skipping pair '{w1}', '{w2}' — one or both not in the vocabulary.)")
        
# Explore most similar words
probe = 'men' if 'men' in word_to_idx else vocab[0]
print(f"\nMost similar to '{probe}':")
for w, s in trainer.most_similar(probe, top_k=10):
    print(f"  {w:>12}  {s:+.3f}")


Training pairs: 432260; each with 5 negatives (or fewer in tiny vocab cases).
Epoch 1/2 — avg loss: 1.8674
Epoch 2/2 — avg loss: 1.8213
cos('love', 'hate') = 0.207
cos('good', 'bad') = 0.202
cos('poor', 'rich') = 0.390
cos('all', 'none') = 0.006
cos('nay', 'no') = 0.384
cos('valiant', 'brave') = 0.367
cos('say', 'tell') = 0.568
cos('small', 'little') = 0.440

Most similar to 'men':
         years  +0.682
         times  +0.648
        myself  +0.565
        strong  +0.563
         worth  +0.528
          them  +0.527
           joy  +0.520
         marks  +0.520
          much  +0.518
         horse  +0.518



## 8. ✍️ Your turn: hyperparameters & observations

Play with these parameters and re‑run training:
- `WINDOW_SIZE` (e.g., 1–5)
- `NEGATIVES_PER_POS` (e.g., 2–10)
- `EMBED_DIM` (e.g., 20–200)
- `EPOCHS` (e.g., 1–5)
- `LR` (learning rate, e.g., 0.005–0.05)

Then write **two observations** here (1–2 sentences each):
1. **Observation 1:** When I increased the WINDOW_SIZE to 4, each word was trained with a wider context. That reduced the number of training pairs and shifted what the embeddings captured. With the smaller window, the model emphasized closer more local relationships, while the larger window added broader contextual information, which changed similarity scores and nearest neighbors.

2. **Observation 2:** I was surprised that the smaller window actually gave a clearer similarity between words like valiant and brave. I would have thought that using more context would make the embeddings better, but it seems like the larger window added extra surrounding words that made the relationship less precise.


## Appendix: How this all fits together (optional reading)

- **Skip‑gram idea:** Given a center word, predict nearby words.  
- **Your part:** You created positive word pairs using a **window**, and **negative words** by sampling at random from the rest of the vocabulary.  
- **Black‑box model:** Uses logistic objectives to pull together true pairs and push apart random negatives.  
- **Cosine similarity:** Measures how aligned two word vectors are. High cosine similarity suggests similar usage in the corpus.

### Troubleshooting
- If you see `ImportError: torch ...`, install PyTorch:
  ```bash
  pip install torch
  ```
- If NLTK's tokenizer fails to download (no internet), the fallback regex tokenizer will be used—it’s simpler but sufficient for this lab.
